# Module 10: Advanced Topics in Fair Value Modeling

## Learning Objectives

By the end of this module, you will be able to:

1. Build hierarchical models for correlated commodity complexes
2. Quantify uncertainty in fair value estimates
3. Augment economic models with machine learning
4. Detect and model market regime changes
5. Analyze forward curves for term structure signals
6. Combine multiple advanced techniques for robust modeling

## Introduction

This module covers sophisticated techniques that go beyond basic fair value modeling:

- **Hierarchical Models**: Leverage correlations across commodity complexes
- **Uncertainty Quantification**: Move beyond point estimates to full distributions
- **ML Augmentation**: Use machine learning to enhance economic models
- **Regime Switching**: Adapt models to changing market conditions
- **Forward Curves**: Extract signals from term structure

These techniques are used by sophisticated trading operations to gain an edge.

In [ ]:
# Standard imports
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Union
from scipy import stats, optimize
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings

# Fair value toolkit
from src.fair_value_toolkit import (
    PointInTimeDataFrame,
    FairValueModel,
    InventoryFairValueModel,
    CostOfCarryModel,
    EnsembleFairValueModel,
    WalkForwardValidator,
    SignalGenerator,
    create_lagged_features,
    create_rolling_features,
)

from data.data_fetchers import create_sample_dataset

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All imports successful")

## 1. Hierarchical Models for Commodity Complexes

### 1.1 The Energy Complex Example

Commodities don't exist in isolation. The energy complex (crude oil, gasoline, diesel, natural gas) exhibits strong correlations. Hierarchical models:

- Pool information across related commodities
- Learn shared parameters (partial pooling)
- Improve estimates for commodities with limited data

In [ ]:
# Create data for energy complex
np.random.seed(42)

# Crude oil (primary commodity)
crude_data = create_sample_dataset('crude_oil', '2020-01-01', '2023-12-31')

# Gasoline (correlated with crude)
gasoline_data = crude_data.copy()
gasoline_data['price'] = crude_data['price'] * 1.3 + np.random.normal(0, 5, len(crude_data))

# Natural gas (less correlated)
natgas_data = crude_data.copy()
natgas_data['price'] = crude_data['price'] * 0.15 + np.random.normal(0, 0.5, len(crude_data))

print("Energy complex data created:")
print(f"  Crude oil: {len(crude_data)} observations")
print(f"  Gasoline: {len(gasoline_data)} observations")
print(f"  Natural gas: {len(natgas_data)} observations")

# Calculate correlations
prices = pd.DataFrame({
    'crude': crude_data['price'],
    'gasoline': gasoline_data['price'],
    'natgas': natgas_data['price']
})

print("\nPrice correlations:")
print(prices.corr())

In [ ]:
# Visualize energy complex
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Normalized prices
ax1 = axes[0]
normalized = prices / prices.iloc[0] * 100
normalized.plot(ax=ax1, linewidth=2)
ax1.set_title('Energy Complex: Normalized Prices (Base 100)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Index (Base 100)')
ax1.legend(['Crude Oil', 'Gasoline', 'Natural Gas'])
ax1.grid(True, alpha=0.3)

# Correlation heatmap
ax2 = axes[1]
sns.heatmap(prices.corr(), annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, ax=ax2, cbar_kws={'label': 'Correlation'})
ax2.set_title('Energy Complex Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Crude and gasoline are highly correlated; natural gas less so.")

### 1.2 Hierarchical Fair Value Model

Instead of separate models for each commodity, use a hierarchical structure:

```
Global parameters (shared across all)
  ↓
Complex parameters (shared within energy)
  ↓
Commodity-specific parameters
```

In [ ]:
class HierarchicalFairValueModel(FairValueModel):
    """
    Hierarchical model for commodity complexes.
    
    Uses partial pooling to share information across related commodities
    while allowing commodity-specific deviations.
    """
    
    def __init__(self, shrinkage: float = 0.5):
        super().__init__(name="HierarchicalFairValueModel")
        self.shrinkage = shrinkage  # 0=no pooling, 1=complete pooling
        self.global_model = None
        self.commodity_models = {}
        self.commodities = []
        
    def fit(self, 
            data_dict: Dict[str, Tuple[pd.DataFrame, pd.Series]],
            as_of_date: Optional[datetime] = None) -> 'HierarchicalFairValueModel':
        """
        Fit hierarchical model.
        
        Parameters
        ----------
        data_dict : dict
            Dictionary mapping commodity name to (X, y) tuples
        """
        self.commodities = list(data_dict.keys())
        
        # Step 1: Fit global model on all data
        X_all = pd.concat([X for X, y in data_dict.values()], axis=0)
        y_all = pd.concat([y for X, y in data_dict.values()], axis=0)
        
        self.global_model = Ridge(alpha=1.0)
        self.global_model.fit(X_all, y_all)
        
        print(f"Global model trained on {len(X_all)} total observations")
        
        # Step 2: Fit commodity-specific models
        for commodity, (X, y) in data_dict.items():
            commodity_model = Ridge(alpha=1.0)
            commodity_model.fit(X, y)
            self.commodity_models[commodity] = commodity_model
            print(f"  {commodity}: {len(X)} observations")
        
        # Step 3: Shrink commodity coefficients toward global
        global_coef = self.global_model.coef_
        
        for commodity, model in self.commodity_models.items():
            # Partial pooling: weighted average of global and commodity-specific
            commodity_coef = model.coef_
            pooled_coef = (self.shrinkage * global_coef + 
                          (1 - self.shrinkage) * commodity_coef)
            model.coef_ = pooled_coef
        
        self.is_fitted = True
        print(f"\n✓ Hierarchical model fitted with shrinkage={self.shrinkage}")
        return self
    
    def predict(self, X: pd.DataFrame, commodity: str) -> pd.DataFrame:
        """
        Predict fair value for specific commodity.
        
        Parameters
        ----------
        X : pd.DataFrame
            Features
        commodity : str
            Commodity name
        """
        if commodity not in self.commodity_models:
            raise ValueError(f"Unknown commodity: {commodity}")
        
        model = self.commodity_models[commodity]
        predictions = model.predict(X)
        
        # Estimate confidence interval (simple approach)
        std_error = np.std(predictions) * 1.96
        
        return pd.DataFrame({
            'fair_value': predictions,
            'lower_bound': predictions - std_error,
            'upper_bound': predictions + std_error,
        }, index=X.index)

print("✓ HierarchicalFairValueModel class defined")

In [ ]:
# Compare independent vs hierarchical models
features = ['inventory', 'production', 'demand']

# Prepare data
data_dict = {
    'crude': (crude_data[features], crude_data['price']),
    'gasoline': (gasoline_data[features], gasoline_data['price']),
    'natgas': (natgas_data[features], natgas_data['price']),
}

# Train hierarchical model
hierarchical_model = HierarchicalFairValueModel(shrinkage=0.3)
hierarchical_model.fit(data_dict)

# Compare predictions for gasoline (benefits from crude information)
X_gas = gasoline_data[features]
y_gas = gasoline_data['price']

# Independent model
independent_model = Ridge(alpha=1.0)
independent_model.fit(X_gas, y_gas)
independent_preds = independent_model.predict(X_gas)

# Hierarchical predictions
hierarchical_preds = hierarchical_model.predict(X_gas, 'gasoline')

# Calculate errors
independent_rmse = np.sqrt(mean_squared_error(y_gas, independent_preds))
hierarchical_rmse = np.sqrt(mean_squared_error(y_gas, hierarchical_preds['fair_value']))

print(f"\nGasoline Model Comparison:")
print(f"  Independent RMSE: {independent_rmse:.3f}")
print(f"  Hierarchical RMSE: {hierarchical_rmse:.3f}")
print(f"  Improvement: {(1 - hierarchical_rmse/independent_rmse)*100:.1f}%")

## 2. Uncertainty Quantification

### 2.1 Beyond Point Estimates

Fair value should be a distribution, not a single number:
- **Parameter uncertainty**: Uncertainty in model coefficients
- **Model uncertainty**: Which model structure is correct?
- **Data uncertainty**: Measurement error in inputs

In [ ]:
class UncertaintyQuantifier:
    """
    Quantify uncertainty in fair value estimates.
    
    Uses bootstrap resampling to estimate parameter uncertainty.
    """
    
    def __init__(self, model_class, n_bootstrap: int = 100):
        self.model_class = model_class
        self.n_bootstrap = n_bootstrap
        self.bootstrap_models = []
        
    def fit(self, X: pd.DataFrame, y: pd.Series) -> 'UncertaintyQuantifier':
        """Fit multiple models via bootstrap."""
        n_samples = len(X)
        
        print(f"Training {self.n_bootstrap} bootstrap models...")
        
        for i in range(self.n_bootstrap):
            # Bootstrap sample
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot = X.iloc[indices]
            y_boot = y.iloc[indices]
            
            # Fit model
            model = self.model_class()
            model.fit(X_boot, y_boot)
            self.bootstrap_models.append(model)
            
            if (i + 1) % 20 == 0:
                print(f"  {i+1}/{self.n_bootstrap} complete")
        
        print("✓ Bootstrap training complete")
        return self
    
    def predict(self, X: pd.DataFrame, 
               quantiles: List[float] = [0.05, 0.25, 0.5, 0.75, 0.95]) -> pd.DataFrame:
        """Generate predictions with quantiles."""
        # Get predictions from all bootstrap models
        all_predictions = []
        for model in self.bootstrap_models:
            preds = model.predict(X)
            all_predictions.append(preds['fair_value'].values)
        
        all_predictions = np.array(all_predictions)
        
        # Calculate quantiles
        result = pd.DataFrame(index=X.index)
        result['mean'] = all_predictions.mean(axis=0)
        result['std'] = all_predictions.std(axis=0)
        
        for q in quantiles:
            result[f'q{int(q*100)}'] = np.quantile(all_predictions, q, axis=0)
        
        return result

print("✓ UncertaintyQuantifier class defined")

In [ ]:
# Example: Uncertainty quantification
X = crude_data[['inventory', 'production', 'demand']]
y = crude_data['price']

# Split data
train_size = int(0.7 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Fit with uncertainty quantification
uq = UncertaintyQuantifier(InventoryFairValueModel, n_bootstrap=50)
uq.fit(X_train, y_train)

# Predict with uncertainty
predictions_uq = uq.predict(X_test)

print("\nPrediction uncertainty:")
print(predictions_uq.head(10))

In [ ]:
# Visualize uncertainty
fig, ax = plt.subplots(figsize=(14, 7))

test_dates = X_test.index

# Plot actual prices
ax.plot(test_dates, y_test, 'o', label='Actual Price', color='black', markersize=4, alpha=0.6)

# Plot mean prediction
ax.plot(test_dates, predictions_uq['mean'], linewidth=2.5, label='Mean Fair Value', color='blue')

# Plot uncertainty bands
ax.fill_between(test_dates, predictions_uq['q5'], predictions_uq['q95'], 
                alpha=0.2, label='90% Confidence', color='blue')
ax.fill_between(test_dates, predictions_uq['q25'], predictions_uq['q75'], 
                alpha=0.3, label='50% Confidence', color='blue')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price ($)', fontsize=12)
ax.set_title('Fair Value with Uncertainty Quantification', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Uncertainty bands show range of plausible fair values.")
print(f"Average uncertainty (90% CI width): ${predictions_uq['q95'].mean() - predictions_uq['q5'].mean():.2f}")

### 2.2 Ensemble Uncertainty

Model uncertainty: which model structure is correct? Use ensemble to capture this.

In [ ]:
# Create ensemble with different model types
models = [
    ('inventory', InventoryFairValueModel(), 0.4),
    ('cost_carry', CostOfCarryModel(), 0.3),
    ('inventory2', InventoryFairValueModel(), 0.3),
]

ensemble = EnsembleFairValueModel(models)
ensemble.fit(X_train, y_train)

# Get predictions from each model
ensemble_preds = ensemble.predict(X_test)

# Get individual model predictions for uncertainty
individual_preds = []
for name, model, weight in models:
    pred = model.predict(X_test)
    individual_preds.append(pred['fair_value'].values)

individual_preds = np.array(individual_preds)

# Calculate model disagreement
model_disagreement = individual_preds.std(axis=0)

print(f"\nModel Uncertainty (disagreement):")
print(f"  Average disagreement: ${model_disagreement.mean():.2f}")
print(f"  Max disagreement: ${model_disagreement.max():.2f}")
print(f"  Min disagreement: ${model_disagreement.min():.2f}")

## 3. Machine Learning Augmentation

### 3.1 Combining Economic Models with ML

Use ML to:
1. Learn non-linear relationships economic models miss
2. Discover feature interactions automatically
3. Adapt to regime changes

**Best practice**: Use economic model as baseline, ML for residuals.

In [ ]:
class MLAugmentedFairValueModel(FairValueModel):
    """
    Fair value model augmented with machine learning.
    
    Uses economic model for interpretable baseline, ML to capture
    complex patterns in residuals.
    """
    
    def __init__(self, base_model, ml_model=None):
        super().__init__(name="MLAugmentedFairValueModel")
        self.base_model = base_model
        self.ml_model = ml_model or GradientBoostingRegressor(
            n_estimators=100,
            max_depth=4,
            learning_rate=0.1,
            random_state=42
        )
        self.scaler = StandardScaler()
        
    def fit(self, X: pd.DataFrame, y: pd.Series, 
           as_of_date: Optional[datetime] = None) -> 'MLAugmentedFairValueModel':
        """Fit base model then ML on residuals."""
        
        # Step 1: Fit economic model
        print("Step 1: Fitting base economic model...")
        self.base_model.fit(X, y, as_of_date)
        
        # Step 2: Calculate residuals
        base_predictions = self.base_model.predict(X)
        residuals = y - base_predictions['fair_value']
        
        base_rmse = np.sqrt(mean_squared_error(y, base_predictions['fair_value']))
        print(f"  Base model RMSE: {base_rmse:.3f}")
        
        # Step 3: Fit ML model to residuals
        print("Step 2: Fitting ML model to residuals...")
        X_scaled = self.scaler.fit_transform(X)
        self.ml_model.fit(X_scaled, residuals)
        
        # Check improvement
        ml_residual_preds = self.ml_model.predict(X_scaled)
        combined_preds = base_predictions['fair_value'] + ml_residual_preds
        combined_rmse = np.sqrt(mean_squared_error(y, combined_preds))
        
        print(f"  Combined model RMSE: {combined_rmse:.3f}")
        print(f"  Improvement: {(1 - combined_rmse/base_rmse)*100:.1f}%")
        
        self.is_fitted = True
        return self
    
    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """Predict using base + ML adjustment."""
        # Base prediction
        base_pred = self.base_model.predict(X)
        
        # ML adjustment
        X_scaled = self.scaler.transform(X)
        ml_adjustment = self.ml_model.predict(X_scaled)
        
        # Combine
        combined_fair_value = base_pred['fair_value'] + ml_adjustment
        
        return pd.DataFrame({
            'fair_value': combined_fair_value,
            'base_fair_value': base_pred['fair_value'],
            'ml_adjustment': ml_adjustment,
            'lower_bound': base_pred['lower_bound'] + ml_adjustment,
            'upper_bound': base_pred['upper_bound'] + ml_adjustment,
        }, index=X.index)
    
    def get_feature_importance(self, feature_names: List[str]) -> pd.DataFrame:
        """Get feature importance from ML model."""
        if hasattr(self.ml_model, 'feature_importances_'):
            importance = pd.DataFrame({
                'feature': feature_names,
                'importance': self.ml_model.feature_importances_
            })
            return importance.sort_values('importance', ascending=False)
        else:
            return None

print("✓ MLAugmentedFairValueModel class defined")

In [ ]:
# Train ML-augmented model
base_model = InventoryFairValueModel()
ml_augmented = MLAugmentedFairValueModel(base_model)

ml_augmented.fit(X_train, y_train)

# Compare on test set
base_test_pred = base_model.predict(X_test)
augmented_test_pred = ml_augmented.predict(X_test)

base_test_rmse = np.sqrt(mean_squared_error(y_test, base_test_pred['fair_value']))
augmented_test_rmse = np.sqrt(mean_squared_error(y_test, augmented_test_pred['fair_value']))

print(f"\n{'='*50}")
print("TEST SET PERFORMANCE")
print(f"{'='*50}")
print(f"Base model RMSE: {base_test_rmse:.3f}")
print(f"ML-augmented RMSE: {augmented_test_rmse:.3f}")
print(f"Improvement: {(1 - augmented_test_rmse/base_test_rmse)*100:.1f}%")
print(f"{'='*50}")

In [ ]:
# Visualize ML adjustments
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

test_dates = X_test.index

# Plot 1: Predictions comparison
ax1.plot(test_dates, y_test, 'o', label='Actual', color='black', markersize=4, alpha=0.6)
ax1.plot(test_dates, base_test_pred['fair_value'], label='Base Model', linewidth=2, alpha=0.7)
ax1.plot(test_dates, augmented_test_pred['fair_value'], label='ML-Augmented', linewidth=2, alpha=0.7)
ax1.set_ylabel('Price ($)')
ax1.set_title('Base Model vs ML-Augmented Model', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: ML adjustments
ax2.bar(test_dates, augmented_test_pred['ml_adjustment'], alpha=0.6, color='green', width=2)
ax2.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('Date')
ax2.set_ylabel('ML Adjustment ($)')
ax2.set_title('Machine Learning Adjustments to Base Model', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 ML model captures patterns missed by economic model.")

### 3.2 Feature Importance Analysis

ML models reveal which features drive residuals.

In [ ]:
# Get feature importance
importance = ml_augmented.get_feature_importance(['inventory', 'production', 'demand'])

print("\nFeature Importance (for residuals):")
print(importance)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance['feature'], importance['importance'], color='steelblue')
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Feature Importance in ML Residual Model', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\n📊 Shows which features help explain what the economic model misses.")

## 4. Regime-Switching Models

### 4.1 Detecting Market Regimes

Markets switch between regimes:
- **Normal regime**: Mean-reverting, stable relationships
- **Crisis regime**: High volatility, broken correlations
- **Contango/Backwardation**: Different storage dynamics

Fair value models should adapt.

In [ ]:
class RegimeDetector:
    """
    Detect market regimes using rolling statistics.
    """
    
    def __init__(self, window: int = 60):
        self.window = window
        
    def detect_regimes(self, prices: pd.Series) -> pd.DataFrame:
        """Detect regimes based on volatility and trend."""
        
        # Calculate features
        returns = prices.pct_change()
        volatility = returns.rolling(self.window).std() * np.sqrt(252)  # Annualized
        trend = prices.rolling(self.window).mean() / prices - 1
        
        # Define regimes
        # High vol threshold: 75th percentile
        high_vol_threshold = volatility.quantile(0.75)
        
        regimes = pd.Series('normal', index=prices.index)
        regimes[volatility > high_vol_threshold] = 'high_volatility'
        regimes[(trend > 0.15) & (volatility <= high_vol_threshold)] = 'bull'
        regimes[(trend < -0.15) & (volatility <= high_vol_threshold)] = 'bear'
        
        return pd.DataFrame({
            'regime': regimes,
            'volatility': volatility,
            'trend': trend,
        })

print("✓ RegimeDetector class defined")

In [ ]:
# Detect regimes in crude oil
regime_detector = RegimeDetector(window=60)
regimes = regime_detector.detect_regimes(crude_data['price'])

# Count regime occurrences
print("\nRegime Distribution:")
print(regimes['regime'].value_counts())

# Visualize regimes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Price with regime coloring
for regime in regimes['regime'].unique():
    mask = regimes['regime'] == regime
    ax1.scatter(regimes.index[mask], crude_data['price'][mask], 
               label=regime, alpha=0.6, s=20)

ax1.plot(crude_data.index, crude_data['price'], color='black', alpha=0.2, linewidth=1)
ax1.set_ylabel('Price ($)')
ax1.set_title('Crude Oil Price Colored by Market Regime', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Volatility
ax2.plot(regimes.index, regimes['volatility'], linewidth=2)
ax2.axhline(regimes['volatility'].quantile(0.75), color='red', 
           linestyle='--', label='High Vol Threshold', alpha=0.7)
ax2.fill_between(regimes.index, 0, regimes['volatility'].quantile(0.75), 
                alpha=0.2, color='green', label='Normal Vol')
ax2.set_xlabel('Date')
ax2.set_ylabel('Annualized Volatility')
ax2.set_title('Rolling Volatility', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Different colors show different market regimes.")

### 4.2 Regime-Conditional Fair Value

Train separate models for each regime.

In [ ]:
class RegimeSwitchingFairValueModel(FairValueModel):
    """
    Fair value model that adapts to market regimes.
    """
    
    def __init__(self, model_class=InventoryFairValueModel):
        super().__init__(name="RegimeSwitchingFairValueModel")
        self.model_class = model_class
        self.regime_models = {}
        self.regime_detector = RegimeDetector()
        
    def fit(self, X: pd.DataFrame, y: pd.Series,
           as_of_date: Optional[datetime] = None) -> 'RegimeSwitchingFairValueModel':
        """Fit separate model for each regime."""
        
        # Detect regimes
        regimes = self.regime_detector.detect_regimes(y)
        
        print("Training regime-specific models...")
        
        # Train model for each regime
        for regime in regimes['regime'].unique():
            mask = regimes['regime'] == regime
            X_regime = X[mask]
            y_regime = y[mask]
            
            if len(X_regime) < 30:  # Skip if too few samples
                print(f"  Skipping {regime}: only {len(X_regime)} samples")
                continue
            
            model = self.model_class()
            model.fit(X_regime, y_regime, as_of_date)
            self.regime_models[regime] = model
            
            print(f"  {regime}: {len(X_regime)} samples")
        
        self.is_fitted = True
        print(f"✓ Trained models for {len(self.regime_models)} regimes")
        return self
    
    def predict(self, X: pd.DataFrame, 
               current_prices: pd.Series) -> pd.DataFrame:
        """Predict using regime-appropriate model."""
        
        # Detect current regimes
        regimes = self.regime_detector.detect_regimes(current_prices)
        
        # Initialize result
        result = pd.DataFrame(index=X.index)
        result['fair_value'] = np.nan
        result['lower_bound'] = np.nan
        result['upper_bound'] = np.nan
        result['regime'] = regimes['regime']
        
        # Predict using regime-specific models
        for regime, model in self.regime_models.items():
            mask = regimes['regime'] == regime
            if mask.sum() > 0:
                preds = model.predict(X[mask])
                result.loc[mask, 'fair_value'] = preds['fair_value'].values
                result.loc[mask, 'lower_bound'] = preds['lower_bound'].values
                result.loc[mask, 'upper_bound'] = preds['upper_bound'].values
        
        # For unseen regimes, use average of all models
        if result['fair_value'].isnull().any():
            all_preds = []
            for model in self.regime_models.values():
                all_preds.append(model.predict(X)['fair_value'].values)
            avg_pred = np.mean(all_preds, axis=0)
            result.loc[result['fair_value'].isnull(), 'fair_value'] = avg_pred[result['fair_value'].isnull()]
        
        return result

print("✓ RegimeSwitchingFairValueModel class defined")

In [ ]:
# Train regime-switching model
regime_model = RegimeSwitchingFairValueModel()
regime_model.fit(X_train, y_train)

# Compare with single model
single_model = InventoryFairValueModel()
single_model.fit(X_train, y_train)

# Predictions
regime_preds = regime_model.predict(X_test, y_test)
single_preds = single_model.predict(X_test)

# Calculate RMSE by regime
print("\nPerformance by Regime:")
print(f"{'='*60}")

for regime in regime_preds['regime'].unique():
    mask = regime_preds['regime'] == regime
    if mask.sum() > 0:
        regime_rmse = np.sqrt(mean_squared_error(
            y_test[mask], regime_preds.loc[mask, 'fair_value']
        ))
        single_rmse = np.sqrt(mean_squared_error(
            y_test[mask], single_preds.loc[mask, 'fair_value']
        ))
        print(f"\n{regime} ({mask.sum()} observations):")
        print(f"  Single model RMSE: {single_rmse:.3f}")
        print(f"  Regime model RMSE: {regime_rmse:.3f}")
        print(f"  Improvement: {(1 - regime_rmse/single_rmse)*100:.1f}%")

print(f"\n{'='*60}")

## 5. Forward Curve Analysis

### 5.1 Term Structure of Fair Value

Instead of a single fair value, estimate the entire forward curve.
The curve shape contains information:
- **Contango** (upward sloping): Storage costs > convenience yield
- **Backwardation** (downward sloping): Scarcity premium
- **Steep slope**: Strong storage signal

In [ ]:
class ForwardCurveFairValueModel(FairValueModel):
    """
    Fair value model for entire forward curve.
    """
    
    def __init__(self, maturities: List[int] = [30, 60, 90, 180, 365]):
        super().__init__(name="ForwardCurveFairValueModel")
        self.maturities = maturities  # Days to maturity
        self.models = {}  # One model per maturity
        
    def fit(self, X: pd.DataFrame, y: pd.Series,
           as_of_date: Optional[datetime] = None) -> 'ForwardCurveFairValueModel':
        """Fit models for each maturity."""
        
        print("Fitting forward curve models...")
        
        for maturity in self.maturities:
            # For simplicity, assume spot plus convenience yield adjustment
            # In practice, use actual forward prices if available
            model = Ridge(alpha=1.0)
            
            # Add maturity as feature
            X_maturity = X.copy()
            X_maturity['maturity_days'] = maturity
            X_maturity['maturity_months'] = maturity / 30
            
            model.fit(X_maturity, y)
            self.models[maturity] = model
            
            print(f"  {maturity} days: fitted")
        
        self.is_fitted = True
        print(f"✓ Forward curve model fitted for {len(self.maturities)} maturities")
        return self
    
    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """Predict fair value for all maturities."""
        
        result = pd.DataFrame(index=X.index)
        
        for maturity in self.maturities:
            X_maturity = X.copy()
            X_maturity['maturity_days'] = maturity
            X_maturity['maturity_months'] = maturity / 30
            
            predictions = self.models[maturity].predict(X_maturity)
            result[f'fv_{maturity}d'] = predictions
        
        return result
    
    def calculate_curve_slope(self, predictions: pd.DataFrame) -> pd.Series:
        """Calculate slope of forward curve."""
        # Slope from 30d to 365d
        if 'fv_30d' in predictions.columns and 'fv_365d' in predictions.columns:
            slope = (predictions['fv_365d'] - predictions['fv_30d']) / 335
            return slope
        return None

print("✓ ForwardCurveFairValueModel class defined")

In [ ]:
# Train forward curve model
curve_model = ForwardCurveFairValueModel(maturities=[30, 60, 90, 180, 365])
curve_model.fit(X_train, y_train)

# Predict curves for test period
curve_predictions = curve_model.predict(X_test)

print("\nForward curve predictions:")
print(curve_predictions.head())

# Calculate slope
curve_slope = curve_model.calculate_curve_slope(curve_predictions)
print(f"\nAverage curve slope: ${curve_slope.mean():.4f} per day")
print(f"Contango (positive slope) periods: {(curve_slope > 0).sum()} / {len(curve_slope)}")
print(f"Backwardation (negative slope) periods: {(curve_slope < 0).sum()} / {len(curve_slope)}")

In [ ]:
# Visualize forward curves at different dates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Select three representative dates
sample_dates = [X_test.index[i] for i in [0, len(X_test)//2, -1]]
colors = ['blue', 'green', 'red']

# Plot 1: Forward curves
ax1 = axes[0]
for date, color in zip(sample_dates, colors):
    curve = curve_predictions.loc[date]
    maturities = [30, 60, 90, 180, 365]
    values = [curve[f'fv_{m}d'] for m in maturities]
    ax1.plot(maturities, values, marker='o', linewidth=2, label=date.strftime('%Y-%m-%d'), color=color)

ax1.set_xlabel('Days to Maturity', fontsize=12)
ax1.set_ylabel('Fair Value ($)', fontsize=12)
ax1.set_title('Forward Curve Shapes', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Curve slope over time
ax2 = axes[1]
ax2.plot(curve_slope.index, curve_slope, linewidth=2, color='purple')
ax2.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax2.fill_between(curve_slope.index, 0, curve_slope, 
                 where=(curve_slope > 0), alpha=0.3, color='blue', label='Contango')
ax2.fill_between(curve_slope.index, 0, curve_slope, 
                 where=(curve_slope < 0), alpha=0.3, color='red', label='Backwardation')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Slope ($/day)', fontsize=12)
ax2.set_title('Forward Curve Slope Over Time', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Curve shape and slope provide additional trading signals.")

### 5.2 Roll Yield Estimation

The forward curve slope implies roll yield for futures traders.

In [ ]:
def estimate_roll_yield(curve_predictions: pd.DataFrame, 
                       holding_period: int = 30) -> pd.Series:
    """
    Estimate roll yield from forward curve.
    
    Roll yield is the return from rolling futures contracts
    in the absence of spot price changes.
    """
    # Roll from front month to next month
    front = curve_predictions['fv_30d']
    back = curve_predictions['fv_60d']
    
    # Roll yield (annualized)
    roll_yield = (front / back - 1) * (365 / holding_period)
    
    return roll_yield

# Calculate roll yield
roll_yield = estimate_roll_yield(curve_predictions)

print("\nRoll Yield Analysis:")
print(f"Average roll yield: {roll_yield.mean():.2%} annualized")
print(f"Positive roll yield: {(roll_yield > 0).sum()} / {len(roll_yield)} periods")
print(f"Negative roll yield: {(roll_yield < 0).sum()} / {len(roll_yield)} periods")

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(roll_yield.index, roll_yield * 100, linewidth=2, color='darkgreen')
ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax.fill_between(roll_yield.index, 0, roll_yield * 100,
               where=(roll_yield > 0), alpha=0.3, color='green', label='Positive Roll')
ax.fill_between(roll_yield.index, 0, roll_yield * 100,
               where=(roll_yield < 0), alpha=0.3, color='red', label='Negative Roll')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Annualized Roll Yield (%)', fontsize=12)
ax.set_title('Estimated Roll Yield from Forward Curve', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Positive roll yield favors long positions; negative favors short.")

## Summary

### Key Takeaways

1. **Hierarchical Models**: Pool information across correlated commodities to improve estimates, especially for data-sparse assets

2. **Uncertainty Quantification**: Always express fair value as a distribution, not a point estimate. Use bootstrap for parameter uncertainty and ensembles for model uncertainty.

3. **ML Augmentation**: Use machine learning to capture complex patterns, but keep economic models as interpretable baseline

4. **Regime Switching**: Markets change—train separate models for different regimes and detect transitions

5. **Forward Curves**: The entire term structure contains information. Curve slope and roll yield are additional signals.

### Integration Strategy

These techniques are most powerful when combined:

```
Base Economic Model
  ↓
+ ML Augmentation (capture non-linearities)
  ↓
+ Regime Switching (adapt to market conditions)
  ↓
+ Hierarchical Structure (share information)
  ↓
+ Uncertainty Quantification (express confidence)
  ↓
Forward Curve Fair Value Distribution
```

### Best Practices

1. **Start Simple**: Begin with basic economic model, add complexity only if it improves out-of-sample performance

2. **Validate Thoroughly**: Use walk-forward testing for all advanced techniques

3. **Monitor Carefully**: Advanced models can fail in subtle ways—monitor all components

4. **Keep Interpretability**: Always maintain an interpretable baseline model

5. **Document Decisions**: Record why you added each layer of complexity

### Next Steps

You've completed the advanced topics module! You're now equipped to:
- Build sophisticated fair value systems
- Quantify and communicate uncertainty
- Adapt models to changing market conditions
- Extract maximum information from data

**Capstone Project**: Build a production-ready fair value system that combines multiple techniques from this course.

## Exercises

1. **Extended Hierarchical Model**: Add more commodities to the energy complex (heating oil, jet fuel) and measure the benefit of hierarchical modeling

2. **Bayesian Uncertainty**: Implement Bayesian regression for uncertainty quantification and compare with bootstrap

3. **Deep Learning**: Try a neural network as the ML augmentation component. Does it improve performance?

4. **Hidden Markov Model**: Implement proper HMM-based regime detection instead of rule-based

5. **Curve Arbitrage**: Design a trading strategy that exploits mispricing in the forward curve shape

6. **Multi-Technique Integration**: Build a model that combines at least three techniques from this module